In [ ]:
# !pip install pymupdf # pdf format
# !pip install pdfplumber # pdf format

In [126]:
import pandas as pd
import re

In [140]:
dataset = "./ABBYY_clear.txt"

In [141]:
with open(dataset, "r") as f: # reading a txt file
    data = f.readlines()

In [142]:
df = pd.DataFrame(data)

In [143]:
df.head()

,0
0,"αάπη, η = αγάπη.\n"
1,"αβά(δ)ωτα, επίρρ. [αβά(δ)ωτος] ανοιχτά, ξεκλεί..."
2,"αβαβίτσα, η βλ. λ. βαβίτσα. ""Πουμέσα η ποκα-λά..."
3,"αβαβοέ, η [παλ. γαλλ. * αν επί - νοου] προκατα..."
4,"αβά(δ)ωτος, ο [στερ. α + βα(δ)ώνω] που δεν κλε..."


In [144]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32829 entries, 0 to 32828
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       32829 non-null  object
dtypes: object(1)
memory usage: 256.6+ KB


In [145]:
# Magic Code (with variants section)
##########################
# df[['word', 'rest']] = df[0].str.extract(r'^([^,]+),\s*(.*)', expand=True)
# df['description'] = df[0].str.extract(r'"(.*?)"', expand=False)
# df['variants'] = df['rest'].str.replace(r'".*?"', '', regex=True).str.strip()
# ################################
# df = df.drop('rest', axis=1)

# df[['word', 'rest']] = df[0].str.extract(r'^([^,]+),\s*(.*)', expand=True)
# df['description'] = df['rest']
# df = df.drop('rest', axis=1)

# df['word'] = df[0].str.extract(r'^([^(=\[\-,]+?)(?=\s*[\(=\[\-,]|$)', expand=False)

# pattern = r'^([^.]+)\.\s+(.*)$|^([^"]+)"\s*(.*)$|^([^/]+)/\s*(.*)$|^([^|]+)\|\s*(.*)$|^([^:]+):\s*(.*)$'
# df['word'] = df[0].str.extract(pattern, expand=False)[0]

df[['word', 'description']] = df[0].str.extract(r'^([^(=\[\-,]+?)(?:\s*[\(=\[\-,]\s*(.*))?$', expand=True)
df = df.drop(0, axis=1)

In [146]:
step_0 = df
step_0.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32829 entries, 0 to 32828
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   word         31795 non-null  object
 1   description  29729 non-null  object
dtypes: object(2)
memory usage: 513.1+ KB


In [149]:
# Cleaning
#########################
df.drop_duplicates(inplace=True)
df = df[~df["word"].str.contains(r'-\d+-', regex=True, na=False)] # -number- deletion
df['description'] = df['description'].fillna('N/A')
#########################

In [150]:
step_1 = df
step_1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31774 entries, 0 to 32828
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   word         31773 non-null  object
 1   description  31774 non-null  object
dtypes: object(2)
memory usage: 744.7+ KB


In [151]:
df = df.dropna(subset=['word']) # NaN for word section

In [152]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31773 entries, 0 to 32828
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   word         31773 non-null  object
 1   description  31773 non-null  object
dtypes: object(2)
memory usage: 744.7+ KB


In [56]:
# ##############################
# # Additional columns for better recognision
# new_columns = ['transcription','greek_analog']
# for col in new_columns:
#     df[col] = ''

# ########################

In [161]:
df[df['description'] == 'N/A'].shape[0] # 2044 left, probably to drop, most of them are just page initials
df.drop(df[df['description'] == 'N/A'].index, inplace=True)

/tmp/ipython-input-161-1296949766.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(df[df['description'] == 'N/A'].index, inplace=True)


In [162]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29729 entries, 0 to 32828
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   word         29729 non-null  object
 1   description  29729 non-null  object
dtypes: object(2)
memory usage: 696.8+ KB


In [155]:
# # df["word"].str.count(',').sum()
# # df["word"].str.count(':').sum()
# # df["word"].str.count(r'\[').sum()

# # print(df["word"].str.contains(',').sum())
# # print(df["word"].str.contains(':').sum()) # 124
# # print(df["word"].str.contains(r'\[').sum())
# # print(df["word"].str.contains(r'\]').sum())
# # print(df["word"].str.contains(r'\(').sum())
# # print(df['word'].str.contains(r'\)').sum()) # 121
# # print(df["word"].str.contains(' - ').sum())
# # print(df["word"].str.contains('–').sum())
# # print(df["word"].str.contains('—').sum())
# # print(df["word"].str.contains(';').sum())
# # print(df["word"].str.contains('\.').sum()) # 1450
# # print(df["word"].str.contains('=').sum())
# # print(df["word"].str.contains('/').sum()) # 525
# # print((df["word"].str.contains(r'\|', na=False)).sum())
# # print(df["word"].str.contains('"').sum()) # 609


# separators = {
#    'dot': df["word"].str.contains('\.', regex=True).sum(),
#    'quotes': df["word"].str.contains('"').sum(),
#    'slash': df["word"].str.contains('/').sum(),
#    'vertical_bar': (df["word"].str.contains(r'\|', na=False)).sum(),
#    'colon': df["word"].str.contains(':').sum(),
#    'round_bracket_open': df["word"].str.contains(r'\(').sum(),
#    'round_bracket_close': df['word'].str.contains(r'\)').sum(),
#    'comma': df["word"].str.contains(',').sum(),
#    'square_bracket_open': df["word"].str.contains(r'\[').sum(),
#    'square_bracket_close': df["word"].str.contains(r'\]').sum(),
#    'dash_with_spaces': df["word"].str.contains(' - ').sum(),
#    'en_dash': df["word"].str.contains('–').sum(),
#    'em_dash': df["word"].str.contains('—').sum(),
#    'semicolon': df["word"].str.contains(';').sum(),
#    'equals': df["word"].str.contains('=').sum()
# }


# df = df[df["word"].str.contains('\.', regex=True)]
# df = df[df["word"].str.contains('"')]
# df = df[df["word"].str.contains('/')]
# df = df[df["word"].str.contains(r'\|', na=False)]
# df = df[df["word"].str.contains(':')]
# df = df[df["word"].str.contains(r'\(')]
# df = df[df['word'].str.contains(r'\)')]
# df = df[df["word"].str.contains(',')]
# df = df[df["word"].str.contains(r'\[')]
# df = df[df["word"].str.contains(r'\]')]

In [173]:
# df[-30:-1]

In [174]:
df.to_csv('clear_cypriot_dict.csv', index=False)